#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

#Reading Data From Silver Table

In [0]:
df = spark.table("workspace.silver.crm_prd_info")
df.printSchema()

# Create a standardized product key for joining Product with Sales

In [0]:
df = df.withColumn(
    "sales_product_key",
    F.concat_ws(
        "-",
        F.element_at(F.split("product_key","-"),-3),
        F.element_at(F.split("product_key","-"),-2),
        F.element_at(F.split("product_key","-"),-1)
    )
)

In [0]:
window_spec = Window.partitionBy("sales_product_key") \
                    .orderBy(F.col("product_start_dt").desc())


In [0]:
df = df.withColumn(
    "rn",
    F.row_number().over(window_spec)
)
df = (
    df.filter(F.col("rn") == 1)
      .drop("rn")
)

In [0]:
df.show()

In [0]:
df.groupBy("sales_product_key").count() \
    .filter(F.col("count") > 1) \
    .show()

#Write into Silver Table

In [0]:
(
    df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("workspace.gold.dim_product")
)